# Notebook 0


# Setup und Helferfunktionen

Dieses Notebook lädt `bench.db` und definiert die Helfer die in den anderen
Kapitel-Notebooks (01–04) benutzt werden. Jedes andere Notebook dupliziert
den Setup-Code, damit es eigenständig läuft d.h. dass man sofort zu anderen Notebooks kann.

## Ausführung

Jedes Notebook lädt `../logs/bench.db` (relativ zum Notebook-Ordner). Die ersten zwei Code-Zellen definieren `raw`, `q()`, `compare()`, `ci95()`, `delta_ci()`, `recommend()` falls man selber etwas checken will.

Voraussetzungen: `pandas`, `matplotlib`, `numpy`, `sqlite3`

## Struktur

| Notebook | Inhalt |
|---|---|
| `00_setup.ipynb` | Setup + Helferfunktionen (dieses hier) |
| `01_workload_and_baseline.ipynb` | Hardware, Datasets, Python-Baseline |
| `02_ceilings.ipynb` | Die vier Hardware-Decken + Multi-Core Skalierung |
| `03_negative_composability.ipynb` | Fünf Negativkomposierung Instanzen |
| `04_dispatch_and_validation.ipynb` | Dispatcher + OOD + Abweichungsprüfung |

## Wie alles abläuft

- 5 Wiederholungen pro (build, threads) Zelle, Median berichtet
- Warm-Cache im ganzen Verlauf, setup wird vor der Messung gewärmt
- Ablationen: `suppress_results=True` zur Isolation der Suchphase
- Python-Baseline + `bestofsuite` End-to-End: `suppress_results=False`
- Perf-Counter (`perf stat`): Median über Wiederholungen, Mittelwert über Instructions
- Ad-hoc Queries: `q(...)`, `compare(...)`, `recommend(...)` sind in jedem Notebook verfügbar, falls es Rückfragen gibt.


In [3]:
# bench.db loading setup
import sqlite3
import re
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

# Hier noch den path ändern, falls bench.db woanders liegt
DB_PATH = Path('../../logs/bench.db')
assert DB_PATH.exists(), f'bench.db not found at {DB_PATH.resolve()}'

with sqlite3.connect(DB_PATH) as _conn:
    raw = pd.read_sql_query('SELECT * FROM runs', _conn)

print(f'loaded {len(raw):,} rows across {raw.build.nunique()} builds, '
      f'{raw.threads.nunique()} thread counts, {raw.label.nunique()} labels')

loaded 2,555 rows across 54 builds, 15 thread counts, 97 labels


In [4]:
# Helferfunktionen für die Analyse der Benchmarks

def parse_dataset(label):
    """Map a bench.db `label` string to a canonical dataset name."""
    if label.startswith('ood_') or 'c1024' in label:
        return 'c1024_56gb'
    if label.startswith('variance_'):
        return 'c256_56gb'
    # Fine-t sweep experiments für c256_56gb
    if label.startswith('lennart_c256_56gb_') or label.startswith('lennart_56gb_'):
        return 'c256_56gb'
    m = re.match(r'(?:main|numa)_(\d+gb)_', label)
    return f'c256_{m.group(1)}' if m else None

raw['dataset'] = raw['label'].apply(parse_dataset)

def q(*, builds=None, threads=None, dataset=None, suppress=True, agg='median'):
    """Slice the runs table by (build, threads, dataset) and aggregate over reps.

    Returns one row per (build, threads) with median wall_s + mean counters.
    """
    df = raw.copy()
    if suppress is not None:
        df = df[df.suppress_results == int(suppress)]
    if builds is not None:
        df = df[df.build.isin(builds if isinstance(builds, (list, tuple)) else [builds])]
    if threads is not None:
        df = df[df.threads.isin(threads if isinstance(threads, (list, tuple)) else [threads])]
    if dataset is not None:
        df = df[df.dataset == dataset]
    if df.empty:
        return df
    metrics = ['wall_s', 'ipc', 'l1_misses', 'llc_misses', 'llc_bw_gbs', 'branch_misses']
    g = df.groupby(['build', 'threads'])
    out = g[metrics].median() if agg == 'median' else g[metrics].mean()
    out['n_reps'] = g.size()
    return out.reset_index().sort_values(['build', 'threads'])

def compare(baseline, ablation, *, threads, dataset, suppress=True):
    """Side-by-side wall + IPC + LLC-miss for baseline vs ablation across t."""
    base = q(builds=[baseline], threads=threads, dataset=dataset, suppress=suppress)
    abl = q(builds=[ablation], threads=threads, dataset=dataset, suppress=suppress)
    if base.empty or abl.empty:
        return pd.DataFrame()
    m = base[['threads','wall_s','ipc','llc_misses']].merge(
        abl[['threads','wall_s','ipc','llc_misses']],
        on='threads', suffixes=(f'_{baseline}', f'_{ablation}'))
    m['ratio_wall'] = m[f'wall_s_{ablation}'] / m[f'wall_s_{baseline}']
    m['delta_ipc'] = m[f'ipc_{ablation}'] - m[f'ipc_{baseline}']
    return m

def ci95(series):
    """5-rep 95% CI half-width via Welch-style t(4) ≈ 2.776."""
    import statistics, math
    n = len(series)
    if n < 2:
        return 0.0
    sd = statistics.stdev(series)
    return 2.776 * sd / math.sqrt(n)

def delta_ci(baseline_walls, ablation_walls):
    """Return (mean_delta_pct, ci95_pct_half) for ablation vs baseline."""
    import statistics, math
    n_a, n_b = len(baseline_walls), len(ablation_walls)
    m_a, m_b = statistics.mean(baseline_walls), statistics.mean(ablation_walls)
    sd_a = statistics.stdev(baseline_walls) if n_a > 1 else 0
    sd_b = statistics.stdev(ablation_walls) if n_b > 1 else 0
    diff = m_b - m_a
    se = math.sqrt(sd_a**2/n_a + sd_b**2/n_b)
    half = 2.776 * se
    return 100*diff/m_a, 100*half/m_a

def datasets_available():
    return raw[raw.dataset.notna()].groupby('dataset').agg(
        n_rows=('id','count'), n_builds=('build','nunique'),
        threads=('threads', lambda s: sorted(s.unique())))

print('helpers loaded: q(), compare(), ci95(), delta_ci(), parse_dataset(), datasets_available()')
datasets_available()

helpers loaded: q(), compare(), ci95(), delta_ci(), parse_dataset(), datasets_available()


,n_rows,n_builds,threads
dataset,,,
c1024_56gb,426,16,"[1, 8, 16, 32, 48, 64, 96]"
c256_10gb,693,20,"[1, 8, 16, 32, 64, 96]"
c256_30gb,546,19,"[1, 8, 16, 32, 64, 96]"
c256_56gb,890,34,"[1, 8, 12, 14, 16, 18, 20, 24, 28, 32, 48, 64,..."
